# H-003 · Bar size (1d vs 1h)

Compare locked pairs under 1D vs 1H after costs. No 4H. Same close-t → open-t+1 contract.
Writes `s2_panel_1h_{train,full}.parquet` once. Freeze `BAR_STAR` on research IS.

Do not open the sealed holdout in this notebook.

Next: run H-004.


## 0. Imports & Config


In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.report import (
    arm_selection_table,
    fold_table,
    fold_val_metrics,
    full_is_metrics,
    load_star_stack,
    median_sharpe_hint,
    plot_fold_boxplots,
    require_star,
    save_star_stack,
    write_tearsheet_pdf,
)
from backtest.s2_coint.research import (
    config_from_stack,
    is_end_for_stack,
    load_s1_weekly,
    load_s2_cuts,
    load_universe_panels,
    lookbacks_for_bar,
    n_trials_local,
    n_trials_stack,
    overlay_ols_hedge,
    overlay_kalman_hedge,
    register_hypothesis_arms,
    s2_artifacts_dir,
    s2_data_dir,
    tearsheet_path,
)
from backtest.s2_coint.runner import run_s2_backtest
from backtest.s2_coint.walkforward import embargo_bars_for_config, make_s2_folds

VERSION = "v2"
STAR_PATH = os.path.join(s2_artifacts_dir(version=VERSION, root=ROOT), "s2_star_stack.json")
LEDGER_PATH = os.path.join(os.path.dirname(STAR_PATH), "s2_variant_ledger.json")
cuts = load_s2_cuts(root=ROOT, version=VERSION)
stack = load_star_stack(STAR_PATH)
print("yahoo_drop_rate", cuts.get("yahoo_drop_rate"))
print("stack keys", sorted(stack.keys()))

from data.ingestion.equity_fetcher import fetch_ohlcv
from data.processing.s2_coint_store import build_pair_panel

require_star("POOL_STAR", stack.get("POOL_STAR"))
require_star("FDR_STAR", stack.get("FDR_STAR"))
PAIRS_STAR = list(stack.get("PAIRS_STAR") or [])
HYP_ID = "H-003"
BAR_ARMS = ("1d", "1h")
DATA_DIR = s2_data_dir(ROOT, version=VERSION)



## 1. Load 1d panel; fetch/build 1h


In [ ]:
train_1d, full_1d = load_universe_panels(None, "1d", PAIRS_STAR, root=ROOT, version=VERSION)
# Tickers from pairs
tickers = sorted({p.split("|")[0] for p in PAIRS_STAR} | {p.split("|")[1] for p in PAIRS_STAR})
ohlc_1h = {}
for t in tickers:
    raw = fetch_ohlcv(t, cuts["PANEL_START"], cuts.get("PANEL_END"), interval="1h", auto_adjust=False)
    if raw is None or raw.empty:
        print("WARN no 1h", t)
        continue
    fr = raw.copy()
    fr["date"] = pd.to_datetime(fr["date"])
    fr = fr.set_index("date")[["open","high","low","close"]].astype(float).sort_index()
    ohlc_1h[t] = fr
pairs = [(p.split("|")[0], p.split("|")[1]) for p in PAIRS_STAR]
panel_1h = build_pair_panel(ohlc_1h, pairs, ols_window=63*6, z_window=60*6, hl_window=252*6, include_adf_pvalue=True)
panel_1h["date"] = pd.to_datetime(panel_1h["date"])
# 70/30 on 1h sessions for train/full files
from backtest.s2_coint.research import overlap_is_end
cut_1h = overlap_is_end(panel_1h, frac=0.70)
train_1h = panel_1h.loc[panel_1h["date"] <= cut_1h].copy()
full_1h = panel_1h.copy()
p_full = os.path.join(DATA_DIR, "s2_panel_1h_full.parquet")
p_train = os.path.join(DATA_DIR, "s2_panel_1h_train.parquet")
full_1h.to_parquet(p_full, index=False)
train_1h.to_parquet(p_train, index=False)
print("wrote", p_train, train_1h.shape, p_full, full_1h.shape)


## 2. Decision table + fold stability


In [ ]:
# Score 1d vs 1h on research IS / train panels after costs (v1 H-002 pattern).
s1_weekly = load_s1_weekly(ROOT)
configs = {arm: config_from_stack(stack, bar=arm, break_mode=stack.get("BREAK_STAR") or "off") for arm in BAR_ARMS}
register_hypothesis_arms(HYP_ID, list(BAR_ARMS), overwrite=True, path=LEDGER_PATH)
parts = []
for arm, panel in (("1d", train_1d), ("1h", train_1h)):
    lb = lookbacks_for_bar(arm)
    p = overlay_ols_hedge(
        panel,
        ols_window=lb["ols_window"],
        z_window=lb["z_window"],
        hl_window=lb["hl_window"],
        adf_window=lb["adf_window"],
    )
    parts.append(full_is_metrics(p, {arm: configs[arm]}, hyp_id=HYP_ID, s1_weekly=s1_weekly))
display(pd.concat(parts, ignore_index=True))


## 3. Type `BAR_STAR`


In [ ]:
BAR_STAR = ""  # type after the table: "1d" | "1h"
require_star("BAR_STAR", BAR_STAR)
stack["BAR_STAR"] = BAR_STAR
save_star_stack(STAR_PATH, stack)
print("wrote", STAR_PATH, "BAR_STAR", BAR_STAR)


## 4. Notes for next hyp

Next: run H-004.
